water data

In [ ]:
import sys
sys.path.append('..')
import myfunction as mf
import pandas as pd
import re




drive_letter = 'D:'
path_part = "/wyy/pyrunning/spdb_sae/env/"
path_data = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/"

path_paper = drive_letter + path_part + "paper/"
path_egle = drive_letter + path_part + "egle/"
path_norman = drive_letter + path_part + "norman/"
path_ucmr = drive_letter + path_part + "ucmr/"
path_gmp = drive_letter + path_part + "gmp/"
path_wqp = drive_letter + path_part + "wqp/"
path_usgs = drive_letter + path_part + "usgs/"
path_fn = drive_letter + path_part + "fn/"
path_swlg = drive_letter + path_part + "swlg/"
path_ukea = drive_letter + path_part + "ukea/"

path_env = drive_letter + path_part


path_encode = "C:/Users/laowu/OneDrive/file/SPDB/"
path_raw = "C:/Users/laowu/OneDrive/file/"

In [ ]:




df_paper = pd.read_csv(path_paper + 'paper.csv')
df_paper['source'] = 'paper'
df_norman = pd.read_csv(path_norman + 'norman.csv')
df_norman['source'] = 'norman'
df_gmp = pd.read_csv(path_gmp + 'gmp.csv')
df_gmp['source'] = 'gmp'
df_egle = pd.read_csv(path_egle + 'egle.csv')
df_egle['source'] = 'egle'


df_sw = pd.concat([df_paper, df_norman, df_gmp, df_egle],axis=0)

df_sw = df_sw[['paid','poid','lon','lat','year','sw_value','n','limit_value','type', 'source']]


df_sw['n'] = df_sw['n'].fillna(1)

df_sw = df_sw[df_sw['lon'].notna()]
df_sw = df_sw[df_sw['sw_value'].notna()]
df_sw.to_csv(path_env + 'sw_raw.csv', index=False)


In [ ]:

df_lr_raw = pd.read_csv(path_env + 'sw_raw.csv')
def calculate_limit_percentage_and_type(df, limits):

    type_counts = df['type'].value_counts()
    type_percentage = df['type'].value_counts(normalize=True) * 100
    type_info = pd.DataFrame({
        'Count': type_counts,
        'Percentage': type_percentage
    })


    result = []
    for limit in limits:
        count = df[df['limit_value'] > limit].shape[0]
        percentage = (count / df.shape[0]) * 100
        result.append({
            'Condition': f'limit_value > {limit}',
            'Count': count,
            'Percentage': percentage
        })
    limit_info = pd.DataFrame(result)

    return type_info, limit_info



type_info, limit_info = calculate_limit_percentage_and_type(df_lr_raw, [0.01, 0.1, 0.5, 1])


print("Type Information:\n", type_info)
print("\nLimit Value Information:\n", limit_info)

Type Information:
    Count  Percentage
0  62054   67.092659
1  30436   32.907341

Limit Value Information:
             Condition  Count  Percentage
0  limit_value > 0.01  22719   24.563737
1   limit_value > 0.1  16019   17.319710
2   limit_value > 0.5   9313   10.069197
3     limit_value > 1   5409    5.848200


### paper

有两部分数据  
第一个是生物数据录入过程中顺便录入的环境数据，由PYcharm写的SPDB_main.py生成  
第二个是那篇文献整理的数据，我们进行了二次校对与整理  
请注意现在的sw_data3与之前的已经不兼容了


In [ ]:
import pandas as pd


df = pd.read_csv(r'D:\wyy\pyrunning\spdb_sae\env\paper\raw\sw_data3.csv')


df['type'] = df['marker'].notna().astype(int)


def get_limit_value(row):
    if isinstance(row['marker'], (int, float)):  # marker列是数值
        return row['marker']
    elif row['marker'] == 'LOD':
        return row['LOD']
    elif row['marker'] == 'LOQ':
        return row['LOQ']
    elif row['marker'] == 'MDL':
        return row['MDL']
    elif row['marker'] == 'MQL':
        return row['MQL']
    return None

df['limit_value'] = df.apply(get_limit_value, axis=1)

df.to_csv(r'D:\wyy\pyrunning\spdb_sae\env\paper\raw\sw_data3_re.csv', index=False)

print("数据处理完成，保存为 sw_data3_modified.csv")

In [ ]:
import pandas as pd
import re

df_all_ed = pd.read_excel(r'D:\wyy\pyrunning\spdb_sae\env\paper\raw\all_ed.xlsx', sheet_name='Sheet1')
df_sw_data = pd.read_csv(r'D:\wyy\pyrunning\spdb_sae\env\paper\raw\sw_data3_re.csv')

df_all_ed = df_all_ed[df_all_ed['unit']=='ng/L']
df_all_ed = df_all_ed[(df_all_ed['value']!='<MRL')&(df_all_ed['value']!='NAND')&(df_all_ed['value'].notna())]



df_all_ed['time'] = df_all_ed['time'].str.replace(" ", "")

df_all_ed['str_year'] = df_all_ed['time'].str.len()

df_all_ed = df_all_ed[df_all_ed['str_year'].isin([4, 6, 9, 13])]


def process_time(row):
    if row['str_year'] == 4:
        return row['time']
    elif row['str_year'] == 6:
        return row['time'][:4]  # 只保留前4个字符
    elif row['str_year'] in [9, 13]:

        split_values = re.split(r'[\/\-\–]', row['time'])

        if len(split_values) == 2 and split_values[0] == split_values[1]:  # 若前后列相等
            if row['str_year'] == 9:
                return split_values[0]  # 9位时，取前列值
            elif row['str_year'] == 13:
                return split_values[0][:4]  # 13位时，取前列值的前4个字符
        return None  # 若前后列不相等，则不保留数据
    return row['time']


df_all_ed['year'] = df_all_ed.apply(process_time, axis=1)
df_all_ed['n']=1
df_all_ed = df_all_ed[['paid','poid','lon','lat','year','value','n','limit_value','type']]
df_sw_data['n'].fillna(1, inplace=True)
df_sw_data = df_sw_data[['paid','poid','lon','lat','year','value','n','limit_value','type']]
df_all_ed.to_csv(path_paper + 'raw/paper_raw_ed.csv',index=False)
df_sw_data.to_csv(path_paper + 'raw/paper_raw_sw.csv',index=False)



C:\Users\laowu\AppData\Local\Temp\ipykernel_32232\527793425.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sw_data = pd.read_csv(r'D:\wyy\pyrunning\spdb_sae\env\paper\raw\sw_data3_re.csv')


In [ ]:
df_sw_part1 = pd.read_csv(path_paper + 'raw/paper_raw_ed.csv')
df_sw_part2 = pd.read_csv(path_paper + 'raw/paper_raw_sw.csv')


df_sw = pd.concat([df_sw_part1, df_sw_part2],axis=0)
df_sw = df_sw.rename(columns={'value':'sw_value'})
list_dup = [48,132,141,178,72,45,216]

df_sw = df_sw[~df_sw['paid'].isin(list_dup)]

df_sw.to_csv(path_paper + 'paper.csv',index=False)

### norman


In [ ]:
import pandas as pd

df_norman = pd.read_csv(path_norman + '/file/norman_raw.csv')


df_norman['lon_num'] = df_norman.apply(lambda x: -x['lon_num'] if x['lon_sign'] == 'West' and x['lon_num'] > 0 else x['lon_num'], axis=1)


df_norman['poname'] = df_norman['poname'].str.replace('PFOSA', 'FOSA')
df_norman['poname'] = df_norman['poname'].str.replace('PFUdA', 'PFUnDA')


po_df = pd.read_excel(path_raw + 'inf.xlsx', sheet_name='po_pfas')
po_dict = dict(zip(po_df['po_name'], po_df['poid']))

df_norman['poid'] = df_norman['poname'].map(po_dict)


df_norman = df_norman[(df_norman['method']=='HPLC-MS or MS/MS')&(df_norman['sw_value']!=0)]


df_norman = df_norman.rename(columns={'lon_num': 'lon', 'lat_num': 'lat'})




df_norman['sw_value'] = df_norman['sw_value'] * 1000
df_norman['lod'] = df_norman['lod'] * 1000
df_norman['loq'] = df_norman['loq'] * 1000


df_norman['n'] = df_norman['num'].replace({'Individual results': 1, 'Aggregate data': 2})


df_norman['type'] = df_norman['describe'].apply(lambda x: 1 if x in ['Less than LoD', 'Less than LoQ'] else (0 if x == 'Individual Value' else None))

df_norman['limit_value'] = df_norman.apply(
    lambda row: row['lod'] if row['describe'] == 'Less than LoD' else (
        row['loq'] if row['describe'] == 'Less than LoQ' else None
    ),
    axis=1  # 按行应用
)
df_norman['paid'] = 1554
df_norman = df_norman[['paid','poid','lon','lat','year','sw_value','n','limit_value','type']]

output_final_path = path_norman + 'norman.csv'
df_norman.to_csv(output_final_path, index=False)


In [ ]:
df_egle_raw = pd.read_csv(path_egle + '1/raw/egle_1_raw.csv')
df_rp = pd.read_excel(path_raw + 'inf.xlsx', sheet_name='rp')
df_po = df_rp[df_rp['type'] == 'po']
po_dict = dict(zip(df_po['NAME'], df_po['ID']))

df_egle_raw['poid'] = df_egle_raw['posname'].map(po_dict)

import pandas as pd

df_egle_raw = df_egle_raw[~df_egle_raw['Flag'].str.contains('J|I|H', na=False)]


df_egle_raw['type'] = df_egle_raw['Flag'].apply(lambda x: 1 if 'B' in str(x) else 0)


df_egle_raw['limit_value'] = df_egle_raw.apply(lambda row: row['Mdl'] if 'B' in str(row['Flag']) else None, axis=1)


df_egle_raw['value'] = df_egle_raw.apply(lambda row: row['value'] / 2 if 'B' in str(row['Flag']) else row['value'], axis=1)

df_egle_raw = df_egle_raw.rename(columns={'Duplicate': 'n', 'value': 'sw_value'})

df_egle_raw['paid'] = 1561

df_egle_raw = df_egle_raw[['paid','poid','lon','lat','year','sw_value','n','limit_value','type']]

df_egle_raw.to_csv(path_egle + 'egle.csv', index=False)

In [ ]:


import pandas as pd

df_gmp = pd.read_csv(path_gmp + 'raw/gmp_pfas.csv')
df_gmp = df_gmp[['Latitude', 'Longitude', 'Analytical method', 'Matrix', 'Parameter', 'Water type', 'Year','LOQ','No. of values','Median']]
df_gmp = df_gmp[df_gmp['Matrix']=='Water']
df_gmp = df_gmp[(df_gmp['Analytical method']=='HPLC-MS')|(df_gmp['Analytical method']=='HPLC-MS-MS')]


df_gmp[['posname', 'unit']] = df_gmp['Parameter'].str.split(' \(', expand=True)
df_gmp['unit'] = df_gmp['unit'].str.replace(')', '')


df_gmp['LOQ'] = df_gmp['LOQ'].astype(float)

df_gmp['type'] = df_gmp.apply(lambda x: 1 if x['Median'] == 0 else 0, axis=1)

df_gmp['limit_value'] = df_gmp.apply(lambda x: x['LOQ'] if x['Median'] == 0 else '', axis=1)



df_gmp['Median'] = df_gmp.apply(lambda x: x['LOQ']/2 if x['Median'] == 0 else x['Median'], axis=1)


def convert_median(row):
    if row['unit'] == 'pg/l':
        return row['Median'] / 1000  # pg/l 转换为 ng/l
    elif row['unit'] == 'pg/m3':
        return row['Median'] / 1000000  # pg/m^3 转换为 ng/l
    else:
        return row['Median']  # ng/l 的情况不变

df_gmp['Median'] = df_gmp.apply(convert_median, axis=1)
def convert_limit(row):
    if row['type'] == 1:
        return row['Median']*2  # pg/l 转换为 ng/l
    else:
        return None  # ng/l 的情况不变
df_gmp['limit_value'] = df_gmp.apply(convert_limit, axis=1)




df_gmp = df_gmp.rename(columns={'Longitude': 'lon', 'Latitude': 'lat', 'Year':'year','No. of values':'n','Median':'sw_value'})

df_rp = pd.read_excel(path_raw + 'inf.xlsx', sheet_name='rp')
df_po = df_rp[df_rp['type'] == 'po']
po_dict = dict(zip(df_po['NAME'], df_po['ID']))

df_gmp['poid'] = df_gmp['posname'].map(po_dict)
df_gmp['paid'] = 1562

df_gmp = df_gmp[['paid','poid','lon','lat','year','sw_value','n','limit_value','type']]

df_gmp = df_gmp[df_gmp['sw_value'].notna()]
df_gmp.to_csv(path_gmp + 'gmp.csv', index=False)


C:\Users\laowu\AppData\Local\Temp\ipykernel_22468\3529747630.py:10: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df_gmp['unit'] = df_gmp['unit'].str.replace(')', '')
